In [1]:
import pandas as pd
import numpy as np

In [9]:
d = pd.read_csv("../processed/mlenergy_energy_by_task.csv")
d

,model_id,nickname,task,gpu_model,weight_precision,total_params_billions,activated_params_billions,energy_per_token_joules,energy_wh_per_1k,avg_power_watts,output_throughput_tokens_per_sec,total_output_tokens,is_stable
0,Qwen/Qwen3-14B,Qwen 3 14B,gpqa,B200,bfloat16,14.0,14.0,1.366263,0.379518,319.788310,234.060505,1391196.0,True
1,Qwen/Qwen3-14B,Qwen 3 14B,gpqa,B200,bfloat16,14.0,14.0,0.780643,0.216845,360.897524,462.308096,1446226.0,True
2,Qwen/Qwen3-14B,Qwen 3 14B,gpqa,B200,bfloat16,14.0,14.0,0.418057,0.116127,422.342314,1010.250937,1367637.0,True
3,Qwen/Qwen3-14B,Qwen 3 14B,gpqa,B200,bfloat16,14.0,14.0,0.291175,0.080882,507.163546,1741.781137,1383914.0,True
4,Qwen/Qwen3-14B,Qwen 3 14B,gpqa,B200,bfloat16,14.0,14.0,0.182575,0.050715,511.258716,2800.260203,1450748.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
833,Qwen/Qwen3-VL-8B-Instruct,Qwen 3 VL 8B Instruct,video-chat,H100,bfloat16,8.0,8.0,0.551851,0.153292,411.223734,745.172151,139322.0,True
834,Qwen/Qwen3-VL-8B-Instruct,Qwen 3 VL 8B Instruct,video-chat,H100,bfloat16,8.0,8.0,0.385818,0.107172,422.770883,1095.778900,139305.0,True
835,Qwen/Qwen3-VL-8B-Instruct,Qwen 3 VL 8B Instruct,video-chat,H100,bfloat16,8.0,8.0,0.279871,0.077742,466.580924,1667.126670,137858.0,True
836,Qwen/Qwen3-VL-8B-Instruct,Qwen 3 VL 8B Instruct,video-chat,H100,bfloat16,8.0,8.0,0.245684,0.068245,479.692845,1952.481706,138013.0,True


In [ ]:
# controls variable - assigns task size & gpu with energy
d = d[(d.task == "lm-arena-chat") & (d.gpu_model == "H100")]
d = d[(d.energy_wh_per_1k > 0) & d.activated_params_billions.notna()]

In [ ]:
# median of each models run for ACTIVATED_params_billions (not activated don't use energy, therefore is not useful)
per_model = d.groupby("nickname").agg(
    active = ("activated_params_billions", "first"),
    energy = ("energy_wh_per_1k", "median"),
).reset_index()

In [16]:
# calibrating constant k (constant to calculate energy usage through physics formula)
per_model["k"] = per_model.energy / per_model.active
k = per_model.k.median()
k

np.float64(0.005426348104575971)

In [ ]:
# the ratio of energy per active params, low and high factors for outlier exclusion
ratio = per_model.energy / (k * per_model.active)
low_factor = ratio.quantile(0.10)
high_factor = ratio.quantile(0.90)

In [18]:
# adding active parameter estimation for closed models, utilizes Epoch's * LifeArchitect public data
master = pd.read_csv("../processed/master_models.csv")
closed = master[master.energy_wh_per_1k.isna()].copy()
closed["active_params_est"] = closed["parameters"] / 1e9

In [21]:
# three estimation for each closed model (low, med, high)
closed["energy_best"] = k * closed.active_params_est
closed["energy_low"]  = closed.energy_best * low_factor
closed["energy_high"] = closed.energy_best * high_factor
closed["source_quality"] = "estimated"
closed["method"] = f"k={k:.5f} Wh/1k/Bactive, H100 chat, ±band from 16 measured models"

In [22]:
print(closed[["model_name", "active_params_est", "energy_low", "energy_best", "energy_high"]].sort_values("energy_best").head(20))


                           model_name  active_params_est    energy_low  \
834            Samuel Neural Checkers       1.600000e-08  3.626363e-11   
8                             ADALINE       1.700000e-08  3.853010e-11   
341                               GNN       3.000000e-08  6.799430e-11   
82                      Bankruptcy-NN       3.600000e-08  8.159316e-11   
819                             SNARC       4.000000e-08  9.065906e-11   
911                           Theseus       4.000000e-08  9.065906e-11   
847            Self Organizing System       2.250000e-07  5.099572e-10   
528            LSTM with forget gates       2.760000e-07  6.255475e-10   
20                            ASE+ACE       3.240000e-07  7.343384e-10   
707            Piecewise linear model       3.570000e-07  8.091321e-10   
249     Distributed representation NN       4.320000e-07  9.791179e-10   
122  Cancer drug mechanism prediction       5.940000e-07  1.346287e-09   
562         MLP with back-propagation 